# 02 — Yield Curves & Discounting

This notebook covers:
- **Flat forward** curves
- **Piecewise bootstrap** from deposit + swap helpers
- **Zero / forward / discount** relationships
- **JAX AD**: differentiate through the bootstrapped curve via `jax.grad`
- **JAX vmap**: build curves for 100 parallel rate scenarios in one call

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, load_quantlib, compare, compare_table, timed_ms, plot_speedup
jax = setup_backend(BACKEND)
import jax.numpy as jnp
QL = load_quantlib()

---
## 1. Flat Forward Curve

The simplest term structure: a **constant continuously-compounded forward rate** $f$.

$$
P(0, T) = e^{-f \cdot T}, \qquad z(T) = f, \qquad f(t, T) = f
$$

In [ ]:
# ── QuantLib ────────────────────────────────────────────────────────────────
ql_today = QL.Date(15, 1, 2024)
QL.Settings.instance().evaluationDate = ql_today

ql_flat = QL.FlatForward(ql_today, 0.05, QL.Actual365Fixed())

tenors = [0.25, 0.5, 1.0, 2.0, 5.0, 10.0, 30.0]
print("QuantLib Flat Forward (5%):")
for t in tenors:
    df = ql_flat.discount(t)
    zr = ql_flat.zeroRate(t, QL.Continuous).rate()
    print(f"  T={t:5.2f}  discount={df:.10f}  zero={zr:.10f}")

In [ ]:
# ── ql-jax ──────────────────────────────────────────────────────────────────
from ql_jax.time.date import Date
from ql_jax.termstructures.yield_.flat_forward import FlatForward

jax_today = Date(15, 1, 2024)
jax_flat = FlatForward(jax_today, 0.05)

print("ql-jax Flat Forward (5%):")
for t in tenors:
    df = float(jax_flat.discount(t))
    zr = float(jax_flat.zero_rate(t))
    print(f"  T={t:5.2f}  discount={df:.10f}  zero={zr:.10f}")

In [ ]:
# ── Comparison ──────────────────────────────────────────────────────────────
rows = []
for t in tenors:
    rows.append((f"discount({t})",
                 ql_flat.discount(t),
                 float(jax_flat.discount(t))))
    rows.append((f"zero_rate({t})",
                 ql_flat.zeroRate(t, QL.Continuous).rate(),
                 float(jax_flat.zero_rate(t))))

compare_table(rows, title="Flat Forward: Discount & Zero Rate")

---
## 2. Piecewise Yield Curve Bootstrap

Build a curve from market deposit and swap rates, solving for discount factors at each pillar.

**Market data** (EUR-like):
| Instrument | Tenor | Rate |
|---|---|---|
| Deposit | 1M | 3.80% |
| Deposit | 3M | 3.90% |
| Deposit | 6M | 4.00% |
| Swap | 1Y | 4.10% |
| Swap | 2Y | 4.20% |
| Swap | 3Y | 4.25% |
| Swap | 5Y | 4.30% |
| Swap | 7Y | 4.35% |
| Swap | 10Y | 4.40% |

In [ ]:
# ── QuantLib Piecewise Curve ────────────────────────────────────────────────
QL.Settings.instance().evaluationDate = ql_today

helpers = []
# Deposits
for tenor, rate in [(QL.Period(1, QL.Months), 0.0380),
                     (QL.Period(3, QL.Months), 0.0390),
                     (QL.Period(6, QL.Months), 0.0400)]:
    helpers.append(QL.DepositRateHelper(
        QL.QuoteHandle(QL.SimpleQuote(rate)),
        tenor, 2, QL.TARGET(), QL.ModifiedFollowing, False, QL.Actual360()))

# Swaps
for tenor, rate in [(QL.Period(1, QL.Years),  0.0410),
                     (QL.Period(2, QL.Years),  0.0420),
                     (QL.Period(3, QL.Years),  0.0425),
                     (QL.Period(5, QL.Years),  0.0430),
                     (QL.Period(7, QL.Years),  0.0435),
                     (QL.Period(10, QL.Years), 0.0440)]:
    helpers.append(QL.SwapRateHelper(
        QL.QuoteHandle(QL.SimpleQuote(rate)),
        tenor, QL.TARGET(), QL.Annual, QL.ModifiedFollowing,
        QL.Thirty360(QL.Thirty360.BondBasis),
        QL.Euribor6M()))

ql_curve = QL.PiecewiseLogLinearDiscount(ql_today, helpers, QL.Actual365Fixed())
ql_curve.enableExtrapolation()

print("QuantLib Piecewise Curve:")
eval_tenors = [0.25, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0]
for t in eval_tenors:
    print(f"  T={t:5.1f}  df={ql_curve.discount(t):.10f}  zero={ql_curve.zeroRate(t, QL.Continuous).rate():.6f}")

In [ ]:
# ── ql-jax Piecewise Curve ──────────────────────────────────────────────────
from ql_jax.termstructures.yield_.piecewise import PiecewiseYieldCurve
from ql_jax.termstructures.yield_.rate_helpers import DepositRateHelper, SwapRateHelper
from ql_jax._util.types import TimeUnit

jax_helpers = []

# Deposit helpers
ref = jax_today
dep_data = [(1, 0.0380), (3, 0.0390), (6, 0.0400)]  # months, rate
for months, rate in dep_data:
    pillar = ref.advance(months, TimeUnit.Months)
    jax_helpers.append(DepositRateHelper(quote=rate, pillar_date=pillar))

# Swap helpers
swap_data = [(12, 0.0410), (24, 0.0420), (36, 0.0425),
             (60, 0.0430), (84, 0.0435), (120, 0.0440)]
for months, rate in swap_data:
    pillar = ref.advance(months, TimeUnit.Months)
    jax_helpers.append(SwapRateHelper(quote=rate, pillar_date=pillar, tenor_months=months))

jax_curve = PiecewiseYieldCurve(jax_today, jax_helpers)

print("ql-jax Piecewise Curve:")
for t in eval_tenors:
    print(f"  T={t:5.1f}  df={float(jax_curve.discount(t)):.10f}  zero={float(jax_curve.zero_rate(t)):.6f}")

In [ ]:
# ── Visual comparison ───────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

ts = np.linspace(0.01, 12.0, 200)
ql_zeros = [ql_curve.zeroRate(float(t), QL.Continuous).rate() for t in ts]
jax_zeros = [float(jax_curve.zero_rate(float(t))) for t in ts]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(ts, ql_zeros, label="QuantLib", linewidth=2)
ax1.plot(ts, jax_zeros, '--', label="ql-jax", linewidth=2)
ax1.set_xlabel("Tenor (years)")
ax1.set_ylabel("Zero Rate")
ax1.set_title("Zero Rate Curve")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Difference
diffs = [abs(q - j) for q, j in zip(ql_zeros, jax_zeros)]
ax2.semilogy(ts, diffs)
ax2.set_xlabel("Tenor (years)")
ax2.set_ylabel("|QL - JAX| zero rate")
ax2.set_title("Absolute Difference")
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

---
## 3. Zero ↔ Discount ↔ Forward Relationships

Given a zero rate curve $z(T)$:

$$
P(0,T) = e^{-z(T) \cdot T}
$$

Instantaneous forward rate:

$$
f(T) = -\frac{\partial \ln P(0,T)}{\partial T} = z(T) + T \cdot z'(T)
$$

In [ ]:
# ── Forward rates via QuantLib ──────────────────────────────────────────────
print("Forward rates:")
rows = []
for t1, t2 in [(1.0, 2.0), (2.0, 3.0), (5.0, 10.0)]:
    ql_fwd = ql_curve.forwardRate(t1, t2, QL.Continuous).rate()
    jax_fwd = float(jax_curve.forward_rate(t1, t2))
    rows.append((f"f({t1:.0f},{t2:.0f})", ql_fwd, jax_fwd))

compare_table(rows, title="Forward Rates")

---
## 4. JAX-Unique: Differentiate Through the Curve

With JAX, we can compute $\frac{\partial P(0, T)}{\partial r_i}$ — the sensitivity of a discount factor to each input market rate — **automatically**.

In [ ]:
# AD sensitivity: dDiscount(5Y) / d(each market rate)
market_rates = jnp.array([0.0380, 0.0390, 0.0400,
                           0.0410, 0.0420, 0.0425,
                           0.0430, 0.0435, 0.0440])
rate_labels = ["Dep1M", "Dep3M", "Dep6M",
               "Sw1Y", "Sw2Y", "Sw3Y", "Sw5Y", "Sw7Y", "Sw10Y"]

dep_months = [1, 3, 6]
swap_months = [12, 24, 36, 60, 84, 120]

def discount_from_rates(rates, T=5.0):
    """Build piecewise curve from rates and return discount(T)."""
    helpers = []
    for i, (m, r) in enumerate(zip(dep_months, rates[:3])):
        pillar = jax_today.advance(m, TimeUnit.Months)
        helpers.append(DepositRateHelper(quote=r, pillar_date=pillar))
    for i, (m, r) in enumerate(zip(swap_months, rates[3:])):
        pillar = jax_today.advance(m, TimeUnit.Months)
        helpers.append(SwapRateHelper(quote=r, pillar_date=pillar, tenor_months=m))
    curve = PiecewiseYieldCurve(jax_today, helpers)
    return curve.discount(T)

# Full Jacobian: sensitivity of df(5Y) to each rate
grad_fn = jax.grad(discount_from_rates)
sensitivities = grad_fn(market_rates)

print("dDiscount(5Y) / d(rate):")
for label, s in zip(rate_labels, sensitivities):
    print(f"  {label:6s}: {float(s):+.6f}")

In [ ]:
# Full Jacobian matrix: discount at multiple tenors w.r.t. all rates
def discount_vector(rates):
    helpers = []
    for m, r in zip(dep_months, rates[:3]):
        pillar = jax_today.advance(m, TimeUnit.Months)
        helpers.append(DepositRateHelper(quote=r, pillar_date=pillar))
    for m, r in zip(swap_months, rates[3:]):
        pillar = jax_today.advance(m, TimeUnit.Months)
        helpers.append(SwapRateHelper(quote=r, pillar_date=pillar, tenor_months=m))
    curve = PiecewiseYieldCurve(jax_today, helpers)
    return jnp.array([curve.discount(float(t)) for t in eval_tenors])

jacobian = jax.jacrev(discount_vector)(market_rates)
print(f"Jacobian shape: {jacobian.shape}  (output tenors × input rates)")

# Visualize as heatmap
from notebooks._common import plot_heatmap
plot_heatmap(
    np.array(jacobian),
    xlabels=rate_labels,
    ylabels=[f"{t:.0f}Y" for t in eval_tenors],
    title="Discount Factor Jacobian: ∂P(0,T) / ∂rₖ",
    fmt=".4f"
)

---
## 5. Performance: Curve Bootstrap

In [ ]:
def build_ql_curve():
    helpers = []
    for tenor, rate in [(QL.Period(1, QL.Months), 0.0380),
                         (QL.Period(3, QL.Months), 0.0390),
                         (QL.Period(6, QL.Months), 0.0400)]:
        helpers.append(QL.DepositRateHelper(
            QL.QuoteHandle(QL.SimpleQuote(rate)),
            tenor, 2, QL.TARGET(), QL.ModifiedFollowing, False, QL.Actual360()))
    for tenor, rate in [(QL.Period(1, QL.Years),  0.0410),
                         (QL.Period(2, QL.Years),  0.0420),
                         (QL.Period(3, QL.Years),  0.0425),
                         (QL.Period(5, QL.Years),  0.0430),
                         (QL.Period(7, QL.Years),  0.0435),
                         (QL.Period(10, QL.Years), 0.0440)]:
        helpers.append(QL.SwapRateHelper(
            QL.QuoteHandle(QL.SimpleQuote(rate)),
            tenor, QL.TARGET(), QL.Annual, QL.ModifiedFollowing,
            QL.Thirty360(QL.Thirty360.BondBasis), QL.Euribor6M()))
    c = QL.PiecewiseLogLinearDiscount(ql_today, helpers, QL.Actual365Fixed())
    c.enableExtrapolation()
    return c.discount(5.0)

def build_jax_curve():
    return float(discount_from_rates(market_rates, T=5.0))

ql_ms, _  = timed_ms(build_ql_curve, warmup=3, runs=10)
jax_ms, _ = timed_ms(build_jax_curve, warmup=3, runs=10)

print(f"Curve bootstrap + discount(5Y):")
print(f"  QuantLib : {ql_ms:.2f} ms")
print(f"  ql-jax   : {jax_ms:.2f} ms")

---
## 6. Exercises

1. Add FRA helpers (3×6, 6×9, 9×12) to the piecewise curve and compare discount factors.
2. Compute the full Jacobian of zero rates (not discount factors) w.r.t. market rates.
3. Use `jax.vmap` to bootstrap 100 curves in parallel from perturbed market rates.

---
*End of Notebook 02*